# Exp2 IVMH / MVHT Breakdown

This notebook reruns a deterministic scan-mix sweep for `SNAP`, `IVMH`, `MONO-WR`, `DUAL-WR`, and `EPOCH-WR`, then plots stacked breakdowns of `InitLoad`, `Update`, `BuildSnap`, `RecentScan`, and either `HistoryScan` or `DeltaScan`.

Trace structure:
- Timed `InitLoad`
- Untimed prelude: `RecentScan -> MarkTs -> Update -> SpecialScan`
- Then `blocks` measured blocks of:
  - `Scan x scans_per_block`
  - `MarkTs`
  - `Update`
- In every other block (2nd, 4th, 6th, ...), replace the first scan with a special scan
- `history` sweep: the special scan is `HistoryScan` reading the readable ts published by the immediately preceding block
- `delta` sweep: the special scan is `DeltaScan` from that readable ts to the current state


In [ ]:
from pathlib import Path
import sys
import importlib
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

ROOT = Path('../../').resolve()
sys.path.append(str(ROOT / 'benches'))
import sigmod_exp_common as _sigmod_exp_common
importlib.reload(_sigmod_exp_common)
sys.path.append(str(ROOT / 'benches' / 'hash_join' / 'htap_simulation'))

from sigmod_exp_common import (
    TOL,
    SIGMOD_BUCKET_NUM,
    SIGMOD_HTAP_WAREHOUSE_COUNT,
    SIGMOD_REPEAT,
    SIGMOD_TRIM,
    SIGMOD_WARMUP,
    apply_paper_style,
    current_run_stamp,
    ensure_dirs,
    run_checked,
)
from bench_script_functions import parse_result

apply_paper_style(ROOT)

EXP_DIR = (ROOT / 'benches' / 'sigmod_exp2_ivmh_history_breakdown').resolve()
DATA_DIR = EXP_DIR / 'data'
FIGS_DIR = EXP_DIR / 'figs'
ensure_dirs(DATA_DIR, FIGS_DIR)

BIN = ROOT / 'target' / 'release' / 'htap_ivmh_history_breakdown_wkld'

ALL_TABLE_SPECS = {
    'snap': {
        'table_type': 'naive',
        'repair_mode': 'no-repair',
        'label': 'SNAP',
    },
    'ivmh': {
        'table_type': 'ivmh',
        'repair_mode': 'no-repair',
        'label': 'IVMH',
    },
    'mono_wr': {
        'table_type': 'heap',
        'repair_mode': 'write-repair',
        'label': 'MONO-WR',
    },
    'dual_wr': {
        'table_type': 'chain',
        'repair_mode': 'write-repair',
        'label': 'DUAL-WR',
    },
    'epoch_wr': {
        'table_type': 'par',
        'repair_mode': 'write-repair',
        'label': 'EPOCH-WR',
    },
}

SIGMOD_SPECIAL_UPDATE_RATIO = 0.0001
SIGMOD_SPECIAL_TRACE_SEED = 232323223
SIGMOD_SPECIAL_BLOCKS = 20
SIGMOD_SPECIAL_SCANS_PER_BLOCK = 5
SIGMOD_SPECIAL_HISTORY_SWEEP = list(range(0, 11, 2))
SIGMOD_SPECIAL_DELTA_SWEEP = list(range(0, 11, 2))
SIGMOD_SPECIAL_SWEEPS = ['history', 'delta']
SIGMOD_SPECIAL_TIMEOUT_SEC = 900
SIGMOD_SPECIAL_STRUCTURES = ['snap', 'ivmh', 'mono_wr', 'dual_wr', 'epoch_wr']

CONFIG = {
    'warehouse_count': SIGMOD_HTAP_WAREHOUSE_COUNT,
    'bucket_num': SIGMOD_BUCKET_NUM,
    'update_ratio': SIGMOD_SPECIAL_UPDATE_RATIO,
    'seed': SIGMOD_SPECIAL_TRACE_SEED,
    'blocks': SIGMOD_SPECIAL_BLOCKS,
    'scans_per_block': SIGMOD_SPECIAL_SCANS_PER_BLOCK,
    'history_scans': SIGMOD_SPECIAL_HISTORY_SWEEP,
    'delta_scans': SIGMOD_SPECIAL_DELTA_SWEEP,
    'sweeps': SIGMOD_SPECIAL_SWEEPS,
    'repeat': SIGMOD_REPEAT,
    'warmup_runs': SIGMOD_WARMUP,
    'trim': SIGMOD_TRIM,
    'timeout_sec': SIGMOD_SPECIAL_TIMEOUT_SEC,
    'structures': SIGMOD_SPECIAL_STRUCTURES,
}

invalid_structures = [key for key in CONFIG['structures'] if key not in ALL_TABLE_SPECS]
if invalid_structures:
    raise ValueError(f'Unknown structure keys: {invalid_structures}')

invalid_sweeps = [name for name in CONFIG['sweeps'] if name not in {'history', 'delta'}]
if invalid_sweeps:
    raise ValueError(f'Unknown sweep names: {invalid_sweeps}')

TABLE_SPECS = [ALL_TABLE_SPECS[key] for key in CONFIG['structures']]

RUN_STAMP = current_run_stamp()
RUN_TAG = '_'.join([
    f"wc{CONFIG['warehouse_count']}",
    f"bn{CONFIG['bucket_num']}",
    f"ur{str(CONFIG['update_ratio']).replace('.', 'p')}",
    f"blk{CONFIG['blocks']}",
    f"spb{CONFIG['scans_per_block']}",
    f"rep{CONFIG['repeat']}",
    f"seed{CONFIG['seed']}",
    RUN_STAMP,
])

COLOR_MAP = {
    'InitLoad': TOL['grey'],
    'Update': TOL['yellow'],
    'BuildSnap': TOL['purple'],
    'HistoryScan': TOL['darkgreen'],
    'DeltaScan': TOL['blue'],
    'RecentScan': TOL['green'],
}
HATCH_MAP = {
    'HistoryScan': '\\',
    'DeltaScan': '\\',
    'RecentScan': '///',
}
STACK_ORDER_FULL = ['InitLoad', 'Update', 'BuildSnap', 'HistoryScan', 'DeltaScan', 'RecentScan']
SWEEP_STACK_ORDER = {
    'history': ['InitLoad', 'Update', 'BuildSnap', 'HistoryScan', 'RecentScan'],
    'delta': ['InitLoad', 'Update', 'BuildSnap', 'DeltaScan', 'RecentScan'],
}
SWEEP_XLABEL = {
    'history': 'Historical Scans (%)',
    'delta': 'Delta Scans (%)',
}
SWEEP_FILE_STEM = {
    'history': 'history',
    'delta': 'delta',
}
SERIES_STYLE = {
    'SNAP': (TOL['red'], ':', 'x'),
    'IVMH': (TOL['yellow'], '--', 'P'),
    'MONO-WR': (TOL['blue'], '-', 'o'),
    'DUAL-WR': (TOL['cyan'], '-', 's'),
    'EPOCH-WR': (TOL['green'], '-', 'D'),
}

print('ROOT       :', ROOT)
print('BIN        :', BIN)
print('OUTDIR     :', DATA_DIR)
print('CONFIG     :', CONFIG)
print('STRUCTURES :', [spec['label'] for spec in TABLE_SPECS])
print('SWEEPS     :', CONFIG['sweeps'])
print('STAMP      :', RUN_STAMP)
print('TAG        :', RUN_TAG)


In [ ]:
run_checked(
    ['cargo', 'build', '--release', '--bin', 'htap_ivmh_history_breakdown_wkld'],
    ROOT,
    timeout=CONFIG['timeout_sec'],
)
print('Built', BIN)


In [ ]:
def build_args(spec, sweep_type: str, special_scans: int):
    return [
        '--table-type', spec['table_type'],
        '--repair-mode', spec['repair_mode'],
        '--warehouse-count', str(CONFIG['warehouse_count']),
        '--update-ratio', str(CONFIG['update_ratio']),
        '--bucket-num', str(CONFIG['bucket_num']),
        '--seed', str(CONFIG['seed']),
        '--blocks', str(CONFIG['blocks']),
        '--scans-per-block', str(CONFIG['scans_per_block']),
        '--sweep-type', sweep_type,
        '--history-scans', str(special_scans),
    ]


def trim_trial_runs(df: pd.DataFrame) -> pd.DataFrame:
    trim = CONFIG['trim']
    if trim <= 0 or df.empty or 'trial' not in df.columns:
        return df
    totals = (
        df.groupby('trial', as_index=False)['duration_ms']
        .sum()
        .sort_values('duration_ms')
    )
    if len(totals) <= 2 * trim:
        return df
    keep = set(totals.iloc[trim:len(totals) - trim]['trial'])
    return df[df['trial'].isin(keep)].copy()


def classify_rows(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['build_reason'] = out['build_reason'].fillna('')
    out['component'] = None
    out.loc[out['tx_type'] == 'InitLoad', 'component'] = 'InitLoad'
    out.loc[out['tx_type'] == 'Update', 'component'] = 'Update'
    out.loc[(out['tx_type'] == 'MarkTs') & (out['build_reason'] != ''), 'component'] = 'BuildSnap'
    out.loc[out['tx_type'] == 'HistoryScan', 'component'] = 'HistoryScan'
    out.loc[out['tx_type'] == 'DeltaScan', 'component'] = 'DeltaScan'
    out.loc[out['tx_type'] == 'RecentScan', 'component'] = 'RecentScan'
    return out


def aggregate_components(df: pd.DataFrame, label: str, sweep_type: str, special_scans: int) -> pd.DataFrame:
    tx_count = df['tx_id'].nunique()
    comp = (
        df[df['component'].notna()]
        .groupby('component', as_index=False)['duration_ms']
        .sum()
    )
    comp['table_label'] = label
    comp['sweep_type'] = sweep_type
    comp['special_scans'] = special_scans
    comp['sweep_pct'] = 100.0 * special_scans / (CONFIG['blocks'] * CONFIG['scans_per_block'])
    comp['duration_ms'] = comp['duration_ms'] / tx_count
    return comp[['table_label', 'sweep_type', 'special_scans', 'sweep_pct', 'component', 'duration_ms']]


def run_case(spec, sweep_type: str, special_scans: int):
    args = [str(BIN), *build_args(spec, sweep_type, special_scans)]
    print(f"{spec['label']} {sweep_type} special_scans = {special_scans}")
    print('cmd =', ' '.join(args))

    for warmup_idx in range(CONFIG['warmup_runs']):
        _ = run_checked(args, ROOT, timeout=CONFIG['timeout_sec'], quiet=True)
        print(f'  warmup {warmup_idx + 1}/{CONFIG["warmup_runs"]}')

    trials = []
    for trial in range(CONFIG['repeat']):
        result = run_checked(args, ROOT, timeout=CONFIG['timeout_sec'], quiet=True)
        df = parse_result(result.stdout, spec['table_type'])
        df['trial'] = trial
        df['table_label'] = spec['label']
        df['sweep_type'] = sweep_type
        df['special_scans'] = special_scans
        df['sweep_pct'] = 100.0 * special_scans / (CONFIG['blocks'] * CONFIG['scans_per_block'])
        trials.append(classify_rows(df))
        print(f'  trial {trial + 1}/{CONFIG["repeat"]}')

    raw = pd.concat(trials, ignore_index=True)
    trimmed = trim_trial_runs(raw)
    agg = aggregate_components(trimmed, spec['label'], sweep_type, special_scans)
    return raw, trimmed, agg


In [ ]:
RAW_RESULTS = {}
TRIMMED_RESULTS = {}
STACK_RESULTS = {}

for sweep_type in CONFIG['sweeps']:
    sweep_values = CONFIG[f'{sweep_type}_scans']
    raw_rows = []
    trimmed_rows = []
    stack_rows = []

    for spec in TABLE_SPECS:
        for special_scans in sweep_values:
            raw_df, trimmed_df, agg_df = run_case(spec, sweep_type, special_scans)
            raw_rows.append(raw_df)
            trimmed_rows.append(trimmed_df)
            stack_rows.append(agg_df)

    raw_df = pd.concat(raw_rows, ignore_index=True)
    trimmed_df = pd.concat(trimmed_rows, ignore_index=True)
    stack_df = pd.concat(stack_rows, ignore_index=True)

    RAW_RESULTS[sweep_type] = raw_df
    TRIMMED_RESULTS[sweep_type] = trimmed_df
    STACK_RESULTS[sweep_type] = stack_df

    stamped_raw_csv = DATA_DIR / f'ivmh_epoch_{sweep_type}_breakdown_raw_{RUN_TAG}.csv'
    latest_raw_csv = DATA_DIR / f'ivmh-epoch-{sweep_type}-breakdown-raw.csv'
    stamped_stack_csv = DATA_DIR / f'ivmh_epoch_{sweep_type}_breakdown_{RUN_TAG}.csv'
    latest_stack_csv = DATA_DIR / f'ivmh-epoch-{sweep_type}-breakdown.csv'

    raw_df.to_csv(stamped_raw_csv, index=False)
    raw_df.to_csv(latest_raw_csv, index=False)
    stack_df.to_csv(stamped_stack_csv, index=False)
    stack_df.to_csv(latest_stack_csv, index=False)

    display(
        stack_df.pivot(index=['table_label', 'special_scans'], columns='component', values='duration_ms')
        .reindex(columns=STACK_ORDER_FULL[::-1], fill_value=0.0)
        .fillna(0.0)
    )
    print('Saved', stamped_raw_csv)
    print('Saved', latest_raw_csv)
    print('Saved', stamped_stack_csv)
    print('Saved', latest_stack_csv)


In [ ]:
def build_component_legend_handles(stack_order):
    handles = []
    labels = []
    for component in stack_order:
        if component in {'InitLoad', 'Update', 'BuildSnap'}:
            patch = Patch(facecolor=COLOR_MAP[component], edgecolor='black', linewidth=0.4)
        else:
            patch = Patch(facecolor='white', edgecolor=COLOR_MAP[component], linewidth=0.9, hatch=HATCH_MAP[component])
        handles.append(patch)
        labels.append(component)
    return handles, labels


def group_pivot(df, sweep_values, stack_order, label):
    return (
        df[df['table_label'] == label]
        .pivot(index='special_scans', columns='component', values='duration_ms')
        .reindex(index=sweep_values, columns=stack_order, fill_value=0.0)
        .fillna(0.0)
    )


def build_series_legend_handles():
    handles = []
    labels = []
    for spec in TABLE_SPECS:
        label = spec['label']
        color, linestyle, marker = SERIES_STYLE[label]
        handles.append(
            Line2D([0], [0], color=color, linestyle=linestyle, marker=marker, linewidth=1.8, markersize=5.5)
        )
        labels.append(label)
    return handles, labels


def render_regular_plot(stack_df: pd.DataFrame, sweep_type: str, sweep_values):
    stack_order = SWEEP_STACK_ORDER[sweep_type]
    component_handles, component_labels = build_component_legend_handles(stack_order)
    series_handles, series_labels = build_series_legend_handles()
    pivots = {spec['label']: group_pivot(stack_df, sweep_values, stack_order, spec['label']) for spec in TABLE_SPECS}
    y_max_bar = max(float(pivot.sum(axis=1).max()) for pivot in pivots.values()) * 1.12

    fig, (ax_bar, ax_line) = plt.subplots(
        2,
        1,
        figsize=(10.2, 6.8),
        gridspec_kw={'height_ratios': [3.1, 1.7]},
    )
    total_scan_count = CONFIG['blocks'] * CONFIG['scans_per_block']
    sweep_pcts = [100.0 * v / total_scan_count for v in sweep_values]
    sweep_pct_labels = [f'{v:g}' for v in sweep_pcts]

    group_x = list(range(len(sweep_values)))
    bar_width = 0.15
    center = (len(TABLE_SPECS) - 1) / 2.0
    offsets = {
        spec['label']: (idx - center) * (bar_width + 0.02)
        for idx, spec in enumerate(TABLE_SPECS)
    }
    totals = {}
    for spec in TABLE_SPECS:
        label = spec['label']
        totals[label] = pivots[label].sum(axis=1).tolist()

    for spec in TABLE_SPECS:
        label = spec['label']
        pivot = pivots[label]
        x = [v + offsets[label] for v in group_x]
        bottom = [0.0 for _ in x]
        for component in stack_order:
            values = pivot[component].tolist()
            if component in {'InitLoad', 'Update', 'BuildSnap'}:
                ax_bar.bar(x, values, bottom=bottom, width=bar_width, color=COLOR_MAP[component], edgecolor='black', linewidth=0.4)
            else:
                ax_bar.bar(x, values, bottom=bottom, width=bar_width, color='white', edgecolor='black', linewidth=0.4)
                ax_bar.bar(x, values, bottom=bottom, width=bar_width, color='none', edgecolor=COLOR_MAP[component], linewidth=0.9, hatch=HATCH_MAP[component])
            bottom = [b + v for b, v in zip(bottom, values)]
        for xi in x:
            ax_bar.text(xi, -0.085, label, ha='center', va='top', transform=ax_bar.get_xaxis_transform(), fontsize=6.6, rotation=90, clip_on=False)

    ax_bar.set_xticks(group_x)
    ax_bar.set_xticklabels(sweep_pct_labels)
    ax_bar.set_ylabel('Duration (ms / tx)')
    ax_bar.set_xlabel(SWEEP_XLABEL[sweep_type])
    ax_bar.set_ylim(0, y_max_bar)
    ax_bar.yaxis.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)
    component_legend = ax_bar.legend(component_handles, component_labels, loc='upper right', framealpha=0.95)
    ax_bar.add_artist(component_legend)
    ax_bar.legend(series_handles, series_labels, loc='upper left', framealpha=0.95, ncol=2)

    for spec in TABLE_SPECS:
        label = spec['label']
        color, linestyle, marker = SERIES_STYLE[label]
        ax_line.plot(
            sweep_pcts,
            totals[label],
            label=label,
            color=color,
            linestyle=linestyle,
            marker=marker,
            linewidth=1.8,
            markersize=5.5,
        )

    ax_line.set_xlabel(SWEEP_XLABEL[sweep_type])
    ax_line.set_ylabel('Total (ms / tx)')
    ax_line.set_xticks(sweep_pcts)
    ax_line.set_xticklabels(sweep_pct_labels)
    ax_line.set_ylim(bottom=0)
    ax_line.yaxis.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)
    ax_line.legend(loc='upper left', framealpha=0.95, ncol=3)

    fig.tight_layout(rect=[0, 0.02, 1, 1])

    stem = SWEEP_FILE_STEM[sweep_type]
    stamped_pdf = FIGS_DIR / f'exp2_{stem}_breakdown_compare_{RUN_TAG}.pdf'
    latest_pdf = FIGS_DIR / f'exp2-{stem}-breakdown-compare.pdf'
    stamped_png = FIGS_DIR / f'exp2_{stem}_breakdown_compare_{RUN_TAG}.png'
    latest_png = FIGS_DIR / f'exp2-{stem}-breakdown-compare.png'
    fig.savefig(stamped_pdf, format='pdf', bbox_inches='tight')
    fig.savefig(latest_pdf, format='pdf', bbox_inches='tight')
    fig.savefig(stamped_png, dpi=220, bbox_inches='tight')
    fig.savefig(latest_png, dpi=220, bbox_inches='tight')
    plt.show()
    print('Saved', stamped_pdf)
    print('Saved', latest_pdf)
    print('Saved', stamped_png)
    print('Saved', latest_png)


for sweep_type in CONFIG['sweeps']:
    render_regular_plot(STACK_RESULTS[sweep_type], sweep_type, CONFIG[f'{sweep_type}_scans'])


In [ ]:
def draw_axis_break(ax_top, ax_bottom, d=0.012):
    kwargs_top = dict(transform=ax_top.transAxes, color='black', clip_on=False, linewidth=0.9)
    ax_top.plot((-d, +d), (-d, +d), **kwargs_top)
    ax_top.plot((1 - d, 1 + d), (-d, +d), **kwargs_top)
    kwargs_bottom = dict(transform=ax_bottom.transAxes, color='black', clip_on=False, linewidth=0.9)
    ax_bottom.plot((-d, +d), (1 - d, 1 + d), **kwargs_bottom)
    ax_bottom.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs_bottom)


def render_broken_plot(stack_df: pd.DataFrame, sweep_type: str, sweep_values):
    stack_order = SWEEP_STACK_ORDER[sweep_type]
    total_scan_count = CONFIG['blocks'] * CONFIG['scans_per_block']
    sweep_pcts = [100.0 * v / total_scan_count for v in sweep_values]
    sweep_pct_labels = [f'{v:g}' for v in sweep_pcts]

    totals = {
        spec['label']: group_pivot(stack_df, sweep_values, stack_order, spec['label']).sum(axis=1).tolist()
        for spec in TABLE_SPECS
    }
    has_snap = 'SNAP' in totals
    non_snap_labels = [label for label in totals if label != 'SNAP']
    non_snap_max = max((max(totals[label]) for label in non_snap_labels), default=0.0)
    snap_min = min(totals['SNAP']) if has_snap else 0.0
    snap_max = max(totals['SNAP']) if has_snap else 0.0

    if not (has_snap and non_snap_max > 0 and snap_min > non_snap_max * 1.22):
        print(f'{sweep_type}: SNAP separation is not large enough for a useful broken-axis line chart.')
        return

    fig = plt.figure(figsize=(7.6, 4.3))
    gs = fig.add_gridspec(2, 1, height_ratios=[0.9, 1.55], hspace=0.08)
    ax_top = fig.add_subplot(gs[0])
    ax_bottom = fig.add_subplot(gs[1], sharex=ax_top)

    for spec in TABLE_SPECS:
        label = spec['label']
        color, linestyle, marker = SERIES_STYLE[label]
        ax_top.plot(
            sweep_pcts,
            totals[label],
            label=label,
            color=color,
            linestyle=linestyle,
            marker=marker,
            linewidth=1.8,
            markersize=5.5,
        )
        ax_bottom.plot(
            sweep_pcts,
            totals[label],
            label=label,
            color=color,
            linestyle=linestyle,
            marker=marker,
            linewidth=1.8,
            markersize=5.5,
        )

    bottom_max = non_snap_max * 1.10
    top_min = max(snap_min * 0.94, bottom_max + max(1.0, non_snap_max * 0.08))
    top_max = snap_max * 1.05
    ax_bottom.set_ylim(0, bottom_max)
    ax_top.set_ylim(top_min, top_max)
    ax_top.spines['bottom'].set_visible(False)
    ax_bottom.spines['top'].set_visible(False)
    ax_top.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
    ax_top.yaxis.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)
    ax_bottom.yaxis.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)
    ax_bottom.set_xlabel(SWEEP_XLABEL[sweep_type])
    ax_bottom.set_ylabel('Total (ms / tx)')
    ax_bottom.set_xticks(sweep_pcts)
    ax_bottom.set_xticklabels(sweep_pct_labels)
    ax_bottom.legend(loc='lower left', bbox_to_anchor=(0.01, 0.02), framealpha=0.95, ncol=2)
    draw_axis_break(ax_top, ax_bottom)
    fig.tight_layout(rect=[0, 0.02, 1, 1])

    stem = SWEEP_FILE_STEM[sweep_type]
    stamped_pdf = FIGS_DIR / f'exp2_{stem}_breakdown_compare_broken_{RUN_TAG}.pdf'
    latest_pdf = FIGS_DIR / f'exp2-{stem}-breakdown-compare-broken.pdf'
    stamped_png = FIGS_DIR / f'exp2_{stem}_breakdown_compare_broken_{RUN_TAG}.png'
    latest_png = FIGS_DIR / f'exp2-{stem}-breakdown-compare-broken.png'
    fig.savefig(stamped_pdf, format='pdf', bbox_inches='tight')
    fig.savefig(latest_pdf, format='pdf', bbox_inches='tight')
    fig.savefig(stamped_png, dpi=220, bbox_inches='tight')
    fig.savefig(latest_png, dpi=220, bbox_inches='tight')
    plt.show()
    print('Saved', stamped_pdf)
    print('Saved', latest_pdf)
    print('Saved', stamped_png)
    print('Saved', latest_png)


for sweep_type in CONFIG['sweeps']:
    render_broken_plot(STACK_RESULTS[sweep_type], sweep_type, CONFIG[f'{sweep_type}_scans'])
